# Boltz-2 Batch — GFET Probe Structures
Predição estrutural 3D de sondas DNA com Boltz-2.  
**Requer GPU:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Célula 1 — Instalar dependências
!pip install boltz py3Dmol matplotlib -q
print("Instalação concluída.")

In [ ]:
# Célula 2 — Upload do ZIP e extracção dos YAMLs
import zipfile, json, csv, subprocess, shutil
import numpy as np
import matplotlib.pyplot as plt
import py3Dmol
from pathlib import Path
from google.colab import files

uploaded   = files.upload()   # seleccionar boltz2_inputs.zip
zip_name   = list(uploaded.keys())[0]
yaml_dir   = Path("yamls")
yaml_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(yaml_dir)
yaml_files = sorted(yaml_dir.glob("*.yaml"))
print(f"{len(yaml_files)} probes encontradas:")
for y in yaml_files:
    print(f"  {y.stem}")

In [ ]:
# Célula 3 — Predições + visualização por probe
out_root = Path("boltz_results")
out_root.mkdir(exist_ok=True)
results  = []

for i, yf in enumerate(yaml_files, 1):
    probe_id = yf.stem
    out_dir  = out_root / probe_id
    sep = chr(45) * 60
    print(f"\n{sep}")
    print(f"[{i}/{len(yaml_files)}] {probe_id}")

    subprocess.run(
        ["boltz", "predict", str(yf),
         "--out_dir",           str(out_dir),
         "--recycling_steps",   "3",
         "--sampling_steps",    "200",
         "--diffusion_samples", "1",
         "--accelerator",       "gpu",
         "--model",             "boltz2"],
        capture_output=True, text=True
    )

    conf_f  = next(out_dir.rglob("confidence_*_model_0.json"), None)
    plddt_f = next(out_dir.rglob("plddt_*_model_0.npz"), None)
    cif_f   = next(out_dir.rglob("*_model_0.cif"), None)

    confidence = ptm = plddt_mean = plddt_arr = None

    if conf_f:
        d          = json.load(open(conf_f))
        confidence = d.get("confidence_score") or d.get("confidence")
        ptm        = d.get("ptm")

    if plddt_f:
        data       = np.load(plddt_f)
        plddt_arr  = data[data.files[0]].flatten()
        plddt_mean = round(float(plddt_arr.mean()), 3)

    status  = "OK" if cif_f else "FAILED"
    c       = confidence or 0
    quality = ("EXCELLENT" if c >= 0.80 else
               "GOOD"      if c >= 0.70 else
               "MODERATE"  if c >= 0.60 else "LOW")

    results.append({
        "probe_id":   probe_id,
        "status":     status,
        "confidence": round(confidence, 3) if confidence else "",
        "ptm":        round(ptm, 3)        if ptm        else "",
        "plddt":      plddt_mean           if plddt_mean else "",
        "quality":    quality,
        "cif_path":   str(cif_f)           if cif_f      else "",
    })

    print(f"  Confidence: {confidence:.3f}  |  pTM: {ptm:.3f}  |  pLDDT: {plddt_mean:.3f}  |  {quality}")

    # Estrutura 3D interactiva
    if cif_f:
        view = py3Dmol.view(width=500, height=380)
        view.addModel(open(cif_f).read(), "cif")
        view.setStyle({"cartoon": {
            "colorscheme": {"prop": "b", "gradient": "roygb", "min": 0.5, "max": 0.9}
        }})
        view.zoomTo()
        view.show()

    # pLDDT por resíduo
    if plddt_arr is not None:
        fig, ax = plt.subplots(figsize=(6, 2))
        colors  = ["red" if v < 0.5 else "orange" if v < 0.7 else "yellow" if v < 0.9 else "green"
                   for v in plddt_arr]
        ax.bar(range(len(plddt_arr)), plddt_arr, color=colors, width=1.0)
        ax.axhline(0.7, color="gray", linestyle="--", linewidth=0.8)
        ax.set_ylim(0, 1)
        ax.set_xlabel("Residuo")
        ax.set_ylabel("pLDDT")
        ax.set_title(f"{probe_id[:45]}  (media={plddt_mean:.3f})")
        plt.tight_layout()
        plt.show()

json.dump(results, open("results_backup.json", "w"))
n_ok = sum(1 for r in results if r["status"] == "OK")
print(f"Concluido: {n_ok}/{len(results)}")

In [ ]:
# Célula 4 — Resumo e download
header = f"{'probe_id':<50} {'conf':>6} {'pTM':>6} {'pLDDT':>7}  quality"
print(header)
print(chr(45) * 82)
for r in results:
    print(f"{r['probe_id']:<50} {str(r['confidence']):>6} {str(r['ptm']):>6} {str(r['plddt']):>7}  {r['quality']}")

with open("boltz2_results_summary.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=results[0].keys())
    w.writeheader()
    w.writerows(results)

shutil.make_archive("boltz2_all_results", "zip", str(out_root))
files.download("boltz2_all_results.zip")
files.download("boltz2_results_summary.csv")
print("Download iniciado.")